# Qwen2.5-0.5B SFT + DPO on Kaggle T4x2

Use this notebook after pushing the repo to GitHub. Set `REPO_URL`, run the setup cells, then run the pilot pipeline. It avoids reinstalling PyTorch because Kaggle already ships a CUDA-compatible Torch build.

In [ ]:
# Change this after pushing your repo to GitHub.
REPO_URL = "https://github.com/YOUR_USERNAME/qwen-dpo-alignment.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/qwen-dpo-alignment"


In [ ]:
import os
from pathlib import Path

if REPO_URL == "https://github.com/YOUR_USERNAME/qwen-dpo-alignment.git":
    raise ValueError("Set REPO_URL to your GitHub repository URL first.")

repo_path = Path(REPO_DIR)
if not repo_path.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!pwd
!ls -la


In [ ]:
# Kaggle already has torch installed. This installs only the lightweight libraries we need,
# then installs this repo in editable mode without resolving dependencies.
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps

import torch, transformers, datasets
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
!nvidia-smi


## Pilot Settings

Start with the small loop. Once it runs cleanly, change `SUBSET_SIZE=200`, `SFT_TRAIN_EXAMPLES=20000`, and `DPO_TRAIN_EXAMPLES=10000`.

In [ ]:
SUBSET_SIZE = 50
SEED = 42
MODEL = "Qwen/Qwen2.5-0.5B"

SFT_TRAIN_EXAMPLES = 2000
SFT_VAL_EXAMPLES = 200
DPO_TRAIN_EXAMPLES = 1000
DPO_VAL_EXAMPLES = 200

SFT_DIR = "outputs/sft-pilot"
DPO_DIR = "outputs/dpo-pilot"
SUBSET_PATH = f"data/alpaca_eval/alpacaeval_{SUBSET_SIZE}_seed{SEED}.json"


## 1. Create AlpacaEval Subset

In [ ]:
!python scripts/make_alpaca_subset.py   --size {SUBSET_SIZE}   --seed {SEED}   --output_dir data/alpaca_eval


## 2. Generate Base Responses

In [ ]:
!python scripts/generate_alpaca.py   --model_name_or_path {MODEL}   --subset_path {SUBSET_PATH}   --output_path outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json   --generator_name qwen2.5-0.5b-base   --batch_size 8   --max_new_tokens 512


## 3. Supervised Fine-Tuning

In [ ]:
!python scripts/train_sft.py   --model_name_or_path {MODEL}   --output_dir {SFT_DIR}   --max_train_examples {SFT_TRAIN_EXAMPLES}   --max_val_examples {SFT_VAL_EXAMPLES}   --per_device_batch_size 2   --gradient_accumulation_steps 16   --eval_steps 50   --save_steps 100


## 4. Generate SFT Responses

In [ ]:
!python scripts/generate_alpaca.py   --model_name_or_path {SFT_DIR}/best   --subset_path {SUBSET_PATH}   --output_path outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json   --generator_name qwen2.5-0.5b-sft   --batch_size 8   --max_new_tokens 512


## 5. DPO Training

This uses both T4s when available: policy on `cuda:0`, frozen reference on `cuda:1`.

In [ ]:
!python scripts/train_dpo.py   --model_name_or_path {SFT_DIR}/best   --output_dir {DPO_DIR}   --max_train_examples {DPO_TRAIN_EXAMPLES}   --max_val_examples {DPO_VAL_EXAMPLES}   --per_device_batch_size 1   --gradient_accumulation_steps 32   --eval_steps 25   --beta 0.1   --learning_rate 1e-6


## 6. Generate DPO Responses

In [ ]:
!python scripts/generate_alpaca.py   --model_name_or_path {DPO_DIR}/best   --subset_path {SUBSET_PATH}   --output_path outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json   --generator_name qwen2.5-0.5b-dpo   --batch_size 8   --max_new_tokens 512


## 7. Build Pairwise Comparisons

In [ ]:
!python scripts/build_pairwise_eval.py   --a_outputs outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json   --b_outputs outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json   --a_name dpo   --b_name sft   --output_path outputs/eval/dpo_vs_sft_pairs.jsonl

!python scripts/build_pairwise_eval.py   --a_outputs outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json   --b_outputs outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json   --a_name dpo   --b_name base   --output_path outputs/eval/dpo_vs_base_pairs.jsonl

!python scripts/build_pairwise_eval.py   --a_outputs outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json   --b_outputs outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json   --a_name sft   --b_name base   --output_path outputs/eval/sft_vs_base_pairs.jsonl


## 8. Optional Local Judge

This loads a separate local judge. If memory is tight, skip this and download `outputs/eval/*_pairs.jsonl` for manual/API judging.

In [ ]:
RUN_LOCAL_JUDGE = False
JUDGE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

if RUN_LOCAL_JUDGE:
    !python scripts/judge_pairwise_local.py       --pairs_path outputs/eval/dpo_vs_sft_pairs.jsonl       --output_path outputs/eval/dpo_vs_sft_judged.jsonl       --judge_model {JUDGE_MODEL}


## 9. Save Artifacts

Kaggle persists `/kaggle/working` output when you save the notebook version. The model checkpoints and eval files are under `outputs/`.

In [ ]:
!find outputs -maxdepth 3 -type f | sort | sed -n '1,200p'
